# Bazaario Analytics — Notebook 3: Platform Visualisation
## IS215 Digital Business — Group 4

### What this notebook does

This notebook turns raw transaction data into **visual charts and insights** that help platform administrators understand how Bazaario is performing. Each chart answers a specific business question:

| # | Chart | Business Question |
|---|-------|------------------|
| 1 | Revenue Trend | Is our revenue growing or declining over time? |
| 2 | Peak Hours | When are customers most active? When should vendors staff up? |
| 3 | Category Performance | Which types of vendors generate the most revenue? |
| 4 | Weekend vs Weekday | How much more do customers spend on weekends? |
| 5 | Payment Methods | What % of customers use cashless vs cash? |
| 6 | Top Vendors | Which specific vendors are earning the most? |
| 7 | Event Comparison | Which pasar malam events are the most successful? |

These visualisations are the same ones shown in the **Admin Dashboard** of the Bazaario web app.

### Load data and set up chart styling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean, professional chart style
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 11})

In [ ]:
# Load 15,000 transactions from the Bazaario platform
# Each row = one purchase at a pasar malam stall
df = pd.read_csv('data/transactions.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"Loaded {len(df):,} transactions")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()} ({df['date'].nunique()} days)")
print(f"Unique customers: {df['customer_id'].nunique()} | Unique vendors: {df['vendor_name'].nunique()} | Events: {df['event_name'].nunique()}")
df.head()

---
## 1. Daily Revenue Trend

**Question**: Is our platform revenue growing, stable, or declining?

The orange area shows daily revenue (which fluctuates a lot day-to-day). The red line is a **7-day moving average** — it smooths out the noise so we can see the underlying trend. If the red line is going up, the platform is growing.

In [ ]:
# Sum up all transactions per day
daily = df.groupby('date')['amount'].sum().reset_index()
daily.columns = ['date', 'revenue']

# 7-day moving average — smooths out daily spikes to show the trend
daily['ma_7'] = daily['revenue'].rolling(7).mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(daily['date'], daily['revenue'], alpha=0.3, color='#FF6B35')
ax.plot(daily['date'], daily['revenue'], color='#FF6B35', alpha=0.5, linewidth=1, label='Daily Revenue')
ax.plot(daily['date'], daily['ma_7'], color='#E91E63', linewidth=2.5, label='7-Day Moving Average')
ax.set_title('Daily Revenue Trend', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue (SGD)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Average daily revenue: ${daily['revenue'].mean():,.2f}")
print(f"Best day: ${daily['revenue'].max():,.2f} | Worst day: ${daily['revenue'].min():,.2f}")

---
## 2. Peak Hours Analysis

**Question**: When are customers most active? When should vendors prepare the most stock?

The orange bars show transaction count per hour. The pink line shows revenue per hour. The peak tells us when the night market is busiest — vendors should have extra staff and stock ready for that window.

In [ ]:
# Count transactions and revenue per hour of the day
hourly = df.groupby('hour').agg(
    transactions=('transaction_id', 'count'),
    revenue=('amount', 'sum'),
).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.bar(hourly['hour'], hourly['transactions'], color='#FF6B35', alpha=0.7, label='Transactions')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Number of Transactions', color='#FF6B35')
ax1.set_xticks(hourly['hour'])
ax1.set_xticklabels([f'{h}:00' for h in hourly['hour']])

ax2 = ax1.twinx()  # second y-axis for revenue
ax2.plot(hourly['hour'], hourly['revenue'], color='#E91E63', linewidth=2.5, marker='o', label='Revenue')
ax2.set_ylabel('Revenue (SGD)', color='#E91E63')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.set_title('Peak Hours — Transactions & Revenue by Hour', fontweight='bold')
plt.tight_layout()
plt.show()

peak = hourly.loc[hourly['transactions'].idxmax()]
print(f"\nBusiest hour: {int(peak['hour'])}:00 - {int(peak['hour'])+1}:00")
print(f"  → {int(peak['transactions']):,} transactions and ${peak['revenue']:,.2f} in revenue")

---
## 3. Vendor Category Performance

**Question**: Which types of vendors generate the most revenue and transactions?

This helps the admin decide which vendor categories to prioritise when allocating stall spaces for future events.

In [ ]:
cat_stats = df.groupby('vendor_category').agg(
    revenue=('amount', 'sum'),
    transactions=('transaction_id', 'count'),
).sort_values('revenue', ascending=True).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
colors = sns.color_palette('YlOrRd', n_colors=len(cat_stats))

# Left: Revenue
axes[0].barh(cat_stats['vendor_category'], cat_stats['revenue'], color=colors)
axes[0].set_title('Total Revenue by Category', fontweight='bold')
axes[0].set_xlabel('Revenue (SGD)')
for i, v in enumerate(cat_stats['revenue']):
    axes[0].text(v + 50, i, f'${v:,.0f}', va='center')

# Right: Transaction count
axes[1].barh(cat_stats['vendor_category'], cat_stats['transactions'], color=colors)
axes[1].set_title('Transaction Count by Category', fontweight='bold')
axes[1].set_xlabel('Number of Transactions')
for i, v in enumerate(cat_stats['transactions']):
    axes[1].text(v + 10, i, f'{v:,}', va='center')

plt.tight_layout()
plt.show()

---
## 4. Weekend vs Weekday

**Question**: How much more do customers spend on weekends compared to weekdays?

This is critical for vendor staffing. If weekends show a big uplift, vendors should prepare more stock and schedule extra helpers.

In [ ]:
comparison = df.groupby('is_weekend').agg(
    total_revenue=('amount', 'sum'),
    avg_order=('amount', 'mean'),
    transactions=('transaction_id', 'count'),
).reset_index()
comparison['label'] = comparison['is_weekend'].map({True: 'Weekend', False: 'Weekday'})

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = ['#FF6B35', '#E91E63']

for ax, col, title, fmt in zip(
    axes,
    ['total_revenue', 'avg_order', 'transactions'],
    ['Total Revenue', 'Average Order Value', 'Transaction Count'],
    ['${:,.0f}', '${:.2f}', '{:,}']
):
    bars = ax.bar(comparison['label'], comparison[col], color=palette, width=0.5)
    ax.set_title(title, fontweight='bold')
    for bar, val in zip(bars, comparison[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                fmt.format(val), ha='center', va='bottom', fontweight='bold')

plt.suptitle('Weekend vs Weekday Performance', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

weekend_avg = comparison[comparison['label'] == 'Weekend']['avg_order'].values[0]
weekday_avg = comparison[comparison['label'] == 'Weekday']['avg_order'].values[0]
uplift = (weekend_avg / weekday_avg - 1) * 100
print(f"Weekend uplift: {uplift:.1f}% higher average order value")
print(f"  → Weekday avg: ${weekday_avg:.2f} | Weekend avg: ${weekend_avg:.2f}")

---
## 5. Payment Method Distribution

**Question**: What percentage of customers pay cashless (wallet/QR) vs cash?

Singapore's Smart Nation initiative targets high cashless adoption. This chart shows how close the platform is to that goal.

In [ ]:
payment = df.groupby('payment_method').agg(
    count=('transaction_id', 'count'),
    revenue=('amount', 'sum'),
).reset_index()
payment['label'] = payment['payment_method'].map({
    'wallet': 'Bazaario Wallet', 'qr_code': 'QR Code Payment', 'cash': 'Cash',
})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#FF6B35', '#E91E63', '#9C27B0']

for ax, col, title in zip(axes, ['count', 'revenue'], ['By Transaction Count', 'By Revenue']):
    ax.pie(payment[col], labels=payment['label'], autopct='%1.1f%%',
           colors=colors, startangle=90, textprops={'fontsize': 11})
    ax.set_title(title, fontweight='bold')

plt.suptitle('Payment Method Distribution', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

cashless_pct = (1 - payment[payment['payment_method'] == 'cash']['count'].values[0] / payment['count'].sum()) * 100
print(f"Cashless adoption rate: {cashless_pct:.1f}%")
print(f"  → {cashless_pct:.0f}% of transactions are wallet or QR code payments")

---
## 6. Top 15 Vendors by Revenue

**Question**: Which specific vendors are earning the most on the platform?

This ranking helps admins identify top performers (for promotion and featured placement) and understand what makes them successful.

In [ ]:
top = df.groupby('vendor_name')['amount'].sum().nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(12, 8))
colors = sns.color_palette('YlOrRd_r', n_colors=len(top))
ax.barh(top.index, top.values, color=colors)
ax.set_title('Top 15 Vendors by Revenue', fontweight='bold')
ax.set_xlabel('Revenue (SGD)')
for i, v in enumerate(top.values):
    ax.text(v + 20, i, f'${v:,.0f}', va='center')
plt.tight_layout()
plt.show()

---
## 7. Event Performance Comparison

**Question**: Which pasar malam events are the most successful?

Orange bars show revenue per event, pink bars show transaction count. Events with high revenue AND high transactions are the strongest. Events with low performance might need better marketing or vendor selection.

In [ ]:
event_stats = df.groupby('event_name').agg(
    revenue=('amount', 'sum'),
    transactions=('transaction_id', 'count'),
    unique_vendors=('vendor_name', 'nunique'),
    unique_customers=('customer_id', 'nunique'),
).sort_values('revenue', ascending=False).reset_index()

fig, ax = plt.subplots(figsize=(14, 7))
x = range(len(event_stats))
width = 0.35

ax.bar([i - width/2 for i in x], event_stats['revenue'], width,
       label='Revenue (SGD)', color='#FF6B35', alpha=0.8)
ax2 = ax.twinx()
ax2.bar([i + width/2 for i in x], event_stats['transactions'], width,
        label='Transactions', color='#E91E63', alpha=0.8)

ax.set_xlabel('Event')
ax.set_ylabel('Revenue (SGD)', color='#FF6B35')
ax2.set_ylabel('Transactions', color='#E91E63')
ax.set_xticks(x)
ax.set_xticklabels(event_stats['event_name'], rotation=30, ha='right', fontsize=9)
ax.set_title('Event Performance Comparison', fontweight='bold')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
plt.tight_layout()
plt.show()

---
## Executive Summary

A quick overview of all key platform metrics in one place — the same numbers shown on the Admin Dashboard.

In [ ]:
print("=" * 70)
print("  BAZAARIO PLATFORM — EXECUTIVE SUMMARY")
print("=" * 70)
print(f"\n  Total Revenue:           ${df['amount'].sum():>12,.2f}")
print(f"  Total Transactions:      {len(df):>12,}")
print(f"  Average Order Value:     ${df['amount'].mean():>12,.2f}")
print(f"  Unique Customers:        {df['customer_id'].nunique():>12,}")
print(f"  Active Vendors:          {df['vendor_name'].nunique():>12,}")
print(f"  Cashless Rate:           {cashless_pct:>11.1f}%")
print(f"  Weekend Uplift:          {uplift:>11.1f}%")
print(f"  Peak Hour:               {int(peak['hour'])}:00 - {int(peak['hour'])+1}:00")
print(f"  Top Category:            {cat_stats.iloc[-1]['vendor_category']}")
print(f"  Top Event:               {event_stats.iloc[0]['event_name']}")
print("=" * 70)
print("\nThese numbers are displayed in the Admin Dashboard of the Bazaario web app.")